In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dim_product = spark.createDataFrame([
    Row(product_id=1, product_name="Laptop",    category="Electronics", brand="Acme",     color="Silver"),
    Row(product_id=2, product_name="Headphones", category="Electronics", brand="Bose",     color="Black"),
    Row(product_id=3, product_name="Coffee Mug", category="Kitchen",     brand="Ikea",     color="White"),
    Row(product_id=4, product_name="Notebook",    category="Office",      brand="Moleskine", color="Red"),
    Row(product_id=5, product_name="Desk Lamp",   category="Office",      brand="Ikea",     color="Brass"),
])

dim_product.display()

# Slowly Changing Dimensions (SCD)

In dimensional modeling, a **slowly changing dimension** is a dimension whose attribute values change over time — slowly, not every transaction. Ralph Kimball defined six classic types (0–6); a seventh is commonly added in practice. Each type answers the question: *when an attribute value changes, what do we do with the old value?*

---

| Type | Name | Strategy | Old value preserved? | Full history? |
|------|------|---------|---------------------|---------------|
| 0 | Retain / Original | Never change the value | ✅ (never overwritten) | ❌ |
| 1 | Overwrite | Replace old with new | ❌ | ❌ |
| 2 | Add New Row | Insert a new row, close out the old | ✅ | ✅ |
| 3 | Add New Column | Keep old + new in separate columns | ✅ (one prior version) | Partial (current + previous) |
| 4 | Mini-Dimension | Rapidly changing attrs split into a separate dimension | N/A (separate table) | ✅ (via mini-dim grouping) |
| 5 | Mini-Dimension + Type 1 Outrigger | Type 4 + Type 1 ref in base dim | Partial | ✅ (via mini-dim) |
| 6 | Hybrid (1 + 2) | Type 2 history + Type 1 overwrite for selected attrs | Partial | ✅ |
| 7 | Dual Type (1 + 2) | Maintain both a Type 1 table and a Type 2 table | Partial | ✅ |

---

## Detailed explanations

### Type 0 — Retain (Original Value)
Never modify the attribute — the value set at insert time is permanent, regardless of later changes in the source. Rarely used, but useful for immutable facts like a customer's original registration date or date of birth.

### Type 1 — Overwrite
Simply update the row in place. No history is kept — the old value is lost. This is the simplest and most common approach when only the current value matters (e.g., correcting a misspelled name).

### Type 2 — Add New Row (Historical Tracking)
Insert a **new row** for the changed record and mark the previous row as expired. This preserves the full history of changes. Each row gets:
* **effective dates** (or timestamps): `valid_from` / `valid_to`
* a **current flag** (`is_current` = true/false)
* a **surrogate key** so every version has a unique primary key

This is the most widely used SCD type in data warehousing because it lets fact rows join to the dimension version that was active at the time of the fact.

### Type 3 — Add New Column
Add a second column to hold the prior value (e.g., `previous_category` alongside `category`). Only one level of history is kept, so this is suitable when you only need to compare the current value to the immediately preceding one.

### Type 4 — Mini-Dimension
When certain attributes change **rapidly** (e.g., number of dependents, income band, credit score), they are split out of the main dimension into a separate *mini-dimension*. Each unique combination of those attributes becomes a row in the mini-dimension, keyed by a surrogate key. The base dimension references the current mini-dimension key (a Type 1 overwrite on that key). Fact rows then reference the mini-dimension surrogate key, capturing history without exploding the base dimension.

### Type 5 — Mini-Dimension + Type 1 Outrigger
A Type 4 mini-dimension, plus the **base dimension also carries a Type 1 copy** of the key mini-dimension attributes as an outrigger (overwritten on change). This lets you group or filter by the *current* value of a rapidly changing attribute directly from the base dimension, while the mini-dimension preserves the full history.

### Type 6 — Hybrid (Type 1 + Type 2)
Combines Type 2 (full history rows with effective dates) **and** Type 1 (overwrite) for a subset of columns. Typical implementation: the dimension has history rows (Type 2), but also maintains a **current-value column** that is always overwritten to the latest value (Type 1) across *all* rows. This lets you group all historical facts by the current value of an attribute, while still preserving the value as it was at each point in time.

### Type 7 — Dual Type 1 + Type 2
Maintain **two physical tables**: one Type 1 table (current state only, overwritten) and one Type 2 table (full history). Fact tables join to the Type 2 table for point-in-time accuracy, while the Type 1 table serves dashboards and queries that only need the latest snapshot. This trades extra storage for query simplicity.

---

## Type 2 example (most common in practice)

Using the `dim_product` table above, if product 3 changes from category *Kitchen* to *Home*, a Type 2 table would look like:

| product_sk (surrogate) | product_id | product_name | category | valid_from | valid_to | is_current |
|------------------------|------------|--------------|----------|------------|----------|------------|
| 301 | 3 | Coffee Mug | Kitchen | 2024-01-01 | 2024-06-30 | false |
| 302 | 3 | Coffee Mug | Home | 2024-07-01 | 9999-12-31 | true |

The surrogate key `product_sk` is unique per version; the natural key `product_id` repeats. Fact rows reference `product_sk` so each fact is associated with the correct version of the product.

> **Reference:** [Slowly changing dimension — Wikipedia](https://en.wikipedia.org/wiki/Slowly_changing_dimension)

In [0]:
# =====================================================================
# SIMULATION OF INCOMING DATA (one change batch from the source system)
# =====================================================================
# A single batch with three kinds of events:
#   * insert -> brand-new products (id 6, 7)
#   * update -> attribute changes for existing products (id 1, 2, 3)
#   * delete -> product removed from the source (id 4)
# Every SCD type below consumes this SAME batch, each in its own way.

changes = spark.createDataFrame([
    # inserts: new records
    Row(product_id=6, product_name="Mechanical Keyboard", category="Electronics", brand="Logitech", color="Black", change_type="insert", change_date="2024-07-01"),
    Row(product_id=7, product_name="Wall Clock", category="Home", brand="Ikea", color="White", change_type="insert", change_date="2024-07-01"),
    # updates: full new snapshot of the record
    Row(product_id=1, product_name="Laptop", category="Computers", brand="Acme", color="Silver", change_type="update", change_date="2024-07-01"),
    Row(product_id=2, product_name="Headphones", category="Electronics", brand="Sony", color="Black", change_type="update", change_date="2024-07-01"),
    Row(product_id=3, product_name="Coffee Mug", category="Home", brand="Ikea", color="White", change_type="update", change_date="2024-07-01"),
    # delete: only the key matters
    Row(product_id=4, product_name=None, category=None, brand=None, color=None, change_type="delete", change_date="2024-07-01"),
]).withColumn("change_date", F.to_date("change_date"))

display(changes)

In [0]:
# =====================================================================
# TYPE 0 — RETAIN (ORIGINAL VALUE)
# =====================================================================
# * inserts -> applied (new rows added once, then never touched again)
# * updates -> IGNORED (original value kept forever)
# * deletes -> IGNORED (row stays; Type 0 never changes anything)

inserts = changes.filter(F.col("change_type") == "insert")

dim_product_t0 = dim_product.unionByName(
    inserts.select("product_id", "product_name", "category", "brand", "color")
)

display(dim_product_t0.orderBy("product_id"))

In [0]:
# =====================================================================
# TYPE 1 — OVERWRITE
# =====================================================================
# * inserts -> added as new rows
# * updates -> existing row overwritten in place (old value lost)
# * deletes -> row removed

inserts = changes.filter(F.col("change_type") == "insert")
updates = changes.filter(F.col("change_type") == "update")
deletes = changes.filter(F.col("change_type") == "delete").select("product_id")

upd = updates.select(
    "product_id",
    F.col("product_name").alias("new_name"),
    F.col("category").alias("new_category"),
    F.col("brand").alias("new_brand"),
    F.col("color").alias("new_color"),
)

dim_product_t1 = (
    dim_product
    .join(upd, "product_id", "left")
    .withColumn("product_name", F.coalesce("new_name", "product_name"))
    .withColumn("category", F.coalesce("new_category", "category"))
    .withColumn("brand", F.coalesce("new_brand", "brand"))
    .withColumn("color", F.coalesce("new_color", "color"))
    .drop("new_name", "new_category", "new_brand", "new_color")
    .join(deletes, "product_id", "left_anti")
    .unionByName(inserts.select("product_id", "product_name", "category", "brand", "color"))
)

display(dim_product_t1.orderBy("product_id"))

In [0]:
# =====================================================================
# TYPE 2 — ADD NEW ROW (FULL HISTORY)
# =====================================================================
# * inserts -> new current rows
# * updates -> old version closed (valid_to = change_date - 1, is_current = false)
#              + new version inserted with a fresh surrogate key
# * deletes -> old version closed, no replacement row

# --- Step 0: give the base dimension Type 2 bookkeeping columns ---
dim_product_t2 = (
    dim_product
    .withColumn("product_sk", (F.row_number().over(Window.orderBy("product_id")) + F.lit(100)).cast("long"))
    .withColumn("valid_from", F.to_date(F.lit("2024-01-01")))
    .withColumn("valid_to", F.to_date(F.lit("9999-12-31")))
    .withColumn("is_current", F.lit(True))
)

# --- Step 1: close every row hit by an update or a delete ---
expired = (
    dim_product_t2
    .join(
        changes.filter(F.col("change_type").isin("update", "delete")).select("product_id", "change_date"),
        "product_id",
        "left",
    )
    .withColumn("is_current", F.when(F.col("change_date").isNotNull(), F.lit(False)).otherwise(F.col("is_current")))
    .withColumn("valid_to", F.when(F.col("change_date").isNotNull(), F.date_sub(F.col("change_date"), 1)).otherwise(F.col("valid_to")))
    .drop("change_date")
)

# --- Step 2: create new versions for inserts and updates ---
max_sk = dim_product_t2.agg(F.max("product_sk")).collect()[0][0]

new_versions = (
    changes
    .filter(F.col("change_type").isin("insert", "update"))
    .withColumn("product_sk", (F.row_number().over(Window.orderBy("product_id")) + F.lit(max_sk)).cast("long"))
    .withColumn("valid_from", F.col("change_date"))
    .withColumn("valid_to", F.to_date(F.lit("9999-12-31")))
    .withColumn("is_current", F.lit(True))
    .drop("change_type", "change_date")
)

dim_product_t2_final = expired.unionByName(new_versions)

display(dim_product_t2_final.orderBy("product_id", "product_sk"))

In [0]:
# =====================================================================
# TYPE 3 — ADD NEW COLUMN (CURRENT + PREVIOUS)
# =====================================================================
# * inserts -> added as new rows (no previous value yet)
# * updates -> old value moved to previous_category, new value becomes current
# * deletes -> row removed

inserts = changes.filter(F.col("change_type") == "insert")
updates = changes.filter(F.col("change_type") == "update")
deletes = changes.filter(F.col("change_type") == "delete").select("product_id")

# --- Step 0: add the "previous value" column ---
dim_product_t3 = dim_product.withColumn("previous_category", F.lit(None).cast("string"))

# --- Step 1: apply updates: current category moves to previous_category ---
upd = updates.select("product_id", F.col("category").alias("new_category"))

dim_product_t3 = (
    dim_product_t3
    .join(upd, "product_id", "left")
    .withColumn("previous_category", F.when(F.col("new_category").isNotNull(), F.col("category")).otherwise(F.col("previous_category")))
    .withColumn("category", F.coalesce("new_category", "category"))
    .drop("new_category")
    .join(deletes, "product_id", "left_anti")
)

# --- Step 2: add inserts (previous_category still unknown) ---
dim_product_t3 = dim_product_t3.unionByName(
    inserts.select("product_id", "product_name", "category", "brand", "color")
    .withColumn("previous_category", F.lit(None).cast("string"))
)

display(dim_product_t3.orderBy("product_id"))

In [0]:
# =====================================================================
# TYPE 4 — MINI-DIMENSION
# =====================================================================
# Rapidly changing attributes (category, brand, color) are split out of
# the base dimension into a mini-dimension: ONE row per unique attribute
# combination. The base dimension keeps only stable attributes plus a
# reference (mini_sk) to the current group — that reference is
# overwritten Type 1-style on change. History lives in the fact table's
# mini_sk references, not in the dimension itself.
#
# * inserts -> new group(s) in the mini-dim + new base row pointing at them
# * updates -> base row re-pointed to the (possibly new) matching group
# * deletes -> base row removed; groups stay (historical facts still use them)

inserts = changes.filter(F.col("change_type") == "insert")
updates = changes.filter(F.col("change_type") == "update")
deletes = changes.filter(F.col("change_type") == "delete").select("product_id")

# --- Step 0: build the mini-dimension from every attribute combination ---
# (existing rows + everything the incoming batch will produce)
all_combos = (
    dim_product.select("category", "brand", "color")
    .unionByName(
        changes.filter(F.col("change_type").isin("insert", "update"))
        .select("category", "brand", "color")
    )
    .distinct()
)

mini_dim_t4 = all_combos.withColumn(
    "mini_sk", (F.row_number().over(Window.orderBy("category", "brand", "color")) + F.lit(1000)).cast("long")
)

# --- Step 1: base dimension = stable attributes + current group reference ---
base_dim_t4 = (
    dim_product.select("product_id", "product_name", "category", "brand", "color")
    .join(mini_dim_t4, ["category", "brand", "color"])
    .select("product_id", "product_name", "mini_sk")
)

# --- Step 2: apply updates (re-point the base row — Type 1 on the reference) ---
upd_sk = (
    updates.select("product_id", "category", "brand", "color")
    .join(mini_dim_t4, ["category", "brand", "color"])
    .select("product_id", F.col("mini_sk").alias("new_mini_sk"))
)

base_dim_t4_final = (
    base_dim_t4
    .join(upd_sk, "product_id", "left")
    .withColumn("mini_sk", F.coalesce("new_mini_sk", "mini_sk"))
    .drop("new_mini_sk")
    .join(deletes, "product_id", "left_anti")
)

# --- Step 3: apply inserts (new base rows pointing at their groups) ---
ins_sk = (
    inserts.select("product_id", "product_name", "category", "brand", "color")
    .join(mini_dim_t4, ["category", "brand", "color"])
    .select("product_id", "product_name", "mini_sk")
)

base_dim_t4_final = base_dim_t4_final.unionByName(ins_sk)

print("Mini-dimension (one row per unique attribute combination):")
display(mini_dim_t4.orderBy("mini_sk"))
print("Base dimension (stable attributes + current group reference):")
display(base_dim_t4_final.orderBy("product_id"))

In [0]:
# =====================================================================
# TYPE 5 — MINI-DIMENSION + TYPE 1 OUTRIGGER
# =====================================================================
# Type 4 (cell above) PLUS the base dimension carries a Type 1 copy of
# the key mini-dimension attributes as an "outrigger" — overwritten on
# change. This lets you group/filter by the CURRENT value of a rapidly
# changing attribute directly from the base dimension, while the
# mini-dimension preserves the full history via mini_sk references.
#
# Requires the Type 4 cell above to have been executed
# (uses base_dim_t4_final and mini_dim_t4).

# --- Outrigger: current group attributes copied into the base dimension ---
# (in a real warehouse these columns are stored and Type 1-overwritten;
#  here they are derived from the current mini_sk reference, same result)
base_dim_t5 = (
    base_dim_t4_final
    .join(mini_dim_t4, "mini_sk")
    .select(
        "product_id",
        "product_name",
        "mini_sk",
        F.col("category").alias("current_category"),
        F.col("brand").alias("current_brand"),
    )
)

display(base_dim_t5.orderBy("product_id"))
# Note: product 3 (Coffee Mug) and product 7 (Wall Clock) share mini_sk —
# both point at the same (Home, Ikea, White) attribute group.

In [0]:
# =====================================================================
# TYPE 6 — HYBRID (TYPE 2 HISTORY + TYPE 1 CURRENT-VALUE COLUMN)
# =====================================================================
# Full history exactly like Type 2 (reusing dim_product_t2_final from the
# Type 2 cell above), PLUS a current_category column that carries the
# LATEST value on EVERY historical row — so old facts can also be grouped
# by the CURRENT attribute value, while the original columns still show
# the value as it was at each point in time.
#
# Requires the Type 2 cell above to have been executed (uses dim_product_t2_final).

# --- Step 1: latest category per product (from the current versions) ---
current_values = (
    dim_product_t2_final
    .filter(F.col("is_current"))
    .select("product_id", F.col("category").alias("current_category"))
)

# --- Step 2: stamp it onto every row, old versions included ---
dim_product_t6_final = dim_product_t2_final.join(current_values, "product_id", "left")

display(dim_product_t6_final.orderBy("product_id", "product_sk"))
# Note: product 3 shows category=Kitchen on the old row and Home on the new,
# but current_category=Home on BOTH rows. Deleted product 4 has no current
# version, so its historical row keeps current_category = null.

In [0]:
# =====================================================================
# TYPE 7 — DUAL TYPE 1 + TYPE 2 (TWO PHYSICAL TABLES)
# =====================================================================
# Both tables are maintained in parallel from the SAME change batch:
#  * dim_product_t1       (Type 1 cell)  -> current snapshot for dashboards
#  * dim_product_t2_final (Type 2 cell)  -> full history for point-in-time joins
#
# * inserts -> added to both tables
# * updates -> overwritten in the Type 1 table; new version row in the Type 2 table
# * deletes -> removed from the Type 1 table; last version closed in the
#              Type 2 table and kept as history
#
# Requires the Type 1 and Type 2 cells above to have been executed
# (uses dim_product_t1 and dim_product_t2_final).

print("Type 1 table — current snapshot (dashboards / latest value):")
display(dim_product_t1.orderBy("product_id"))
print("Type 2 table — full history (point-in-time fact joins):")
display(dim_product_t2_final.orderBy("product_id", "product_sk"))